# DermoGraph-XAI — DenseNet121 Baseline
**Team 8 | VIT Bhopal | Skin Lesion Classification**

| | |
|---|---|
| Model | DenseNet121 (ImageNet pretrained) |
| Dataset | ALL 6 datasets — HAM10000 + PAD-UFES + ISIC2020 + Melanoma + MIDAS + Derm7pt |
| Classes | 7 (Melanoma, Nevi, BCC, AK, BKL, DF, Vasc) |
| Loss | Weighted Cross Entropy + Focal |
| Optimizer | AdamW + CosineAnnealingLR |
| Special | Dense skip connections — best for medical imaging |

### Benchmark So Far
| Model | Acc | F1 | AUC |
|---|---|---|---|
| VGG16 | 80.48% | 0.7102 | 0.9601 |
| ResNet50 | 87.40% | 0.7261 | 0.9823 |
| **DenseNet121** | **?** | **?** | **?** |

## Cell 1 — Install & Verify GPU

In [ ]:
!pip install -q timm albumentations

import torch
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print("✓ GPU ready" if torch.cuda.is_available() else "✗ NO GPU — go to Session options → GPU T4")


## Cell 2 — Imports & Config

In [ ]:
import os, json, time, warnings, glob, shutil
warnings.filterwarnings('ignore')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

from sklearn.metrics import (f1_score, roc_auc_score,
                              confusion_matrix, classification_report)

# ── Config ────────────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT      = '/kaggle/working/'
CACHE_DIR   = '/kaggle/working/cache/'
BATCH_SIZE  = 32
N_CLASSES   = 7
N_EPOCHS    = 50
LR          = 1e-4
WEIGHT_DECAY= 1e-2
PATIENCE    = 15
GRAD_ACCUM  = 4
MODEL_NAME  = 'densenet121'
TIMM_STR    = 'densenet121'   # timm model string
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]
CLASS_NAMES = ['Melanoma','Nevi','Basal Cell Carcinoma',
               'Actinic Keratosis','Benign Keratosis',
               'Dermatofibroma','Vascular']

torch.manual_seed(42)
np.random.seed(42)
os.makedirs(OUTPUT,    exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"✓ Config ready")
print(f"  Device     : {DEVICE}")
print(f"  Model      : {MODEL_NAME}")
print(f"  Classes    : {N_CLASSES}")
print(f"  Batch size : {BATCH_SIZE} (effective {BATCH_SIZE * GRAD_ACCUM} with grad accum)")
print(f"  Epochs     : {N_EPOCHS}")


## Cell 3 — Load ALL 6 Datasets & Remap Paths

In [ ]:
# ── All 6 Kaggle dataset paths ────────────────────────────
HAM1   = '/kaggle/input/datasets/akshat23029/dermograph-ham-images/HAM10000_images_part_1'
HAM2   = '/kaggle/input/datasets/akshat23029/dermograph-ham-images/HAM10000_images_part_2'
PAD    = '/kaggle/input/datasets/akshat23029/dermograph-pad-images/images'
SPLITS = '/kaggle/input/datasets/akshat23029/dermograph-splits'
ISIC   = '/kaggle/input/datasets/akshat23029/dermograph-isic2020'
MELAN  = '/kaggle/input/datasets/akshat23029/dermograph-melanoma-cancer/melanoma_cancer_dataset'
MIDAS  = '/kaggle/input/datasets/akshat23029/dermograph-midas/midasmultimodalimagedatasetforaibasedskincancer'

# Verify all paths
print("Checking dataset paths...")
all_ok = True
for name, path in [
    ('HAM part1',       HAM1),
    ('HAM part2',       HAM2),
    ('PAD images',      PAD),
    ('Splits CSV',      SPLITS),
    ('ISIC 2020',       ISIC),
    ('Melanoma Cancer', MELAN),
    ('MIDAS',           MIDAS),
]:
    exists = os.path.exists(path)
    n      = len([f for f in os.listdir(path) if not f.startswith('.')]) if exists else 0
    print(f"  {'✓' if exists else '✗'} {name:<20}: {n:,} files")
    if not exists: all_ok = False

assert all_ok, "One or more datasets missing! Add via + Add Input on the right panel"

# ── Load pre-split CSVs ───────────────────────────────────
train_df = pd.read_csv(f'{SPLITS}/train.csv')
val_df   = pd.read_csv(f'{SPLITS}/val.csv')
test_df  = pd.read_csv(f'{SPLITS}/test.csv')

with open(f'{SPLITS}/class_weights.json') as f:
    cw_raw = json.load(f)

print(f"\nSplit sizes: Train={len(train_df):,}  Val={len(val_df):,}  Test={len(test_df):,}")

# ── Build lookups for multi-dir datasets ─────────────────
pad_lookup   = {}
for fname in os.listdir(PAD):
    if fname.lower().endswith(('.jpg','.jpeg','.png')):
        pad_lookup[fname] = f"{PAD}/{fname}"

isic_lookup  = {}
for fname in os.listdir(ISIC):
    if fname.lower().endswith(('.jpg','.jpeg','.png')):
        isic_lookup[fname] = f"{ISIC}/{fname}"

midas_lookup = {}
for fname in os.listdir(MIDAS):
    if fname.lower().endswith(('.jpg','.jpeg','.png','.JPG','.JPEG')):
        midas_lookup[fname] = f"{MIDAS}/{fname}"

print(f"\nLookup tables:")
print(f"  PAD-UFES  : {len(pad_lookup):,}")
print(f"  ISIC 2020 : {len(isic_lookup):,}")
print(f"  MIDAS     : {len(midas_lookup):,}")

# ── Remap Mac paths → Kaggle paths ───────────────────────
def remap(mac_path):
    p     = str(mac_path)
    fname = os.path.basename(p)
    if 'HAM10000_images_part_1' in p:
        fp = f"{HAM1}/{fname}"
        return fp if os.path.exists(fp) else None
    if 'HAM10000_images_part_2' in p:
        fp = f"{HAM2}/{fname}"
        return fp if os.path.exists(fp) else None
    if 'PAD-UFES' in p or 'pad' in p.lower():
        return pad_lookup.get(fname)
    if 'ISIC 2020' in p or 'isic' in p.lower():
        return isic_lookup.get(fname)
    if 'melanoma_cancer_dataset' in p:
        for split in ['train','test']:
            for cls in ['malignant','benign']:
                fp = f"{MELAN}/{split}/{cls}/{fname}"
                if os.path.exists(fp): return fp
        return None
    if 'MIDAS' in p or 'midas' in p.lower():
        return midas_lookup.get(fname)
    return None

print("\nRemapping paths...")
for name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df['kpath'] = df['image_path'].apply(remap)
    before = len(df)
    df.drop(df[df['kpath'].isna()].index, inplace=True)
    df.drop(df[df['label'] >= N_CLASSES].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    src = df['source'].value_counts().to_dict()
    print(f"  {name:<6}: {len(df):>6,} images (dropped {before-len(df)}) | {src}")


## Cell 4 — Cache Preprocessed Images (Hair Removal + Resize)
> **Reuses cache from ResNet50 if you ran that notebook first — will skip already-cached images instantly.**

In [ ]:
def remove_hair(img_bgr):
    """Blackhat morphological hair removal + Telea inpainting."""
    gray     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (11, 11))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask  = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dk       = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    mask     = cv2.dilate(mask, dk, iterations=1)
    return cv2.inpaint(img_bgr, mask, 3, cv2.INPAINT_TELEA)

all_paths = list(set(
    train_df['kpath'].tolist() +
    val_df['kpath'].tolist()   +
    test_df['kpath'].tolist()
))

already = len([f for f in os.listdir(CACHE_DIR) if f.endswith('.jpg')])
print(f"Cache dir    : {CACHE_DIR}")
print(f"Already cached: {already:,} images")
print(f"To process   : {len(all_paths):,} images total")
print("(Skips already-cached images — fast if ResNet50 cache exists)\n")

done = skipped = errors = 0
for src_path in tqdm(all_paths, desc="Caching"):
    fname    = os.path.basename(src_path)
    uid      = str(abs(hash(src_path)))[:10]
    dst_path = f"{CACHE_DIR}{uid}_{fname}"
    if os.path.exists(dst_path):
        skipped += 1
        continue
    img = cv2.imread(src_path)
    if img is None:
        errors += 1
        continue
    img = remove_hair(img)
    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    done += 1

print(f"\n✓ Done: {done:,} new | {skipped:,} skipped | {errors} errors")

# Update paths to cache
cache_lookup = {}
for fname in os.listdir(CACHE_DIR):
    uid = fname[:10]
    cache_lookup[uid] = f"{CACHE_DIR}{fname}"

def to_cache(p):
    uid = str(abs(hash(p)))[:10]
    return cache_lookup.get(uid, p)

for df in [train_df, val_df, test_df]:
    df['kpath'] = df['kpath'].apply(to_cache)

total, used, free = shutil.disk_usage(CACHE_DIR)
print(f"Cache size : {used/1e9:.2f} GB | Free: {free/1e9:.1f} GB")
print("✓ All paths updated to cache")


## Cell 5 — Dataset Class & Augmentation

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=180, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32,
                        max_width=32, p=0.3),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class DermDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(str(row['kpath']))
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(int(row['label']), dtype=torch.long)

# Quick sanity check
img, lbl = DermDataset(train_df, get_train_transforms())[0]
print(f"✓ Dataset OK")
print(f"  Image shape : {img.shape}")
print(f"  Label       : {lbl.item()} = {CLASS_NAMES[lbl.item()]}")


## Cell 6 — DataLoaders & Class Weights

In [ ]:
from torch.utils.data import WeightedRandomSampler

# ── Class weights (from saved JSON, same as ResNet50) ─────
cw_list = [cw_raw[str(i)] for i in range(N_CLASSES)]
cw_list = [min(w, 10.0) for w in cw_list]   # cap at 10x
class_weights = torch.tensor(cw_list, dtype=torch.float32).to(DEVICE)

print("Class weights (capped at 10x):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, cw_list)):
    bar = '█' * int(w * 2)
    print(f"  {i} {name:<25} {w:.2f}  {bar}")

# ── Weighted sampler for minority classes ─────────────────
train_labels  = train_df['label'].values
sample_w      = np.array(cw_list)[train_labels]
sampler       = WeightedRandomSampler(
    weights     = torch.tensor(sample_w, dtype=torch.float32),
    num_samples = len(train_df),
    replacement = True
)

# ── DataLoaders ───────────────────────────────────────────
train_loader = DataLoader(
    DermDataset(train_df, get_train_transforms()),
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = 2,
    pin_memory  = True,
)
val_loader = DataLoader(
    DermDataset(val_df, get_val_transforms()),
    batch_size  = BATCH_SIZE * 2,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True,
)
test_loader = DataLoader(
    DermDataset(test_df, get_val_transforms()),
    batch_size  = BATCH_SIZE * 2,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True,
)

print(f"\n✓ DataLoaders ready")
print(f"  Train batches : {len(train_loader)}")
print(f"  Val   batches : {len(val_loader)}")
print(f"  Test  batches : {len(test_loader)}")


## Cell 7 — Build DenseNet121 Model

In [ ]:
# ── DenseNet121: pretrained backbone + custom head ────────
# DenseNet121 has 1024-dim feature output before classifier
# Dense connections make it excellent for medical imaging
model = timm.create_model(
    TIMM_STR,
    pretrained   = True,
    num_classes  = 0,    # remove default head
    global_pool  = 'avg'
)

# Custom classification head with dropout
class DenseNet121Classifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        feat_dim      = backbone.num_features   # 1024 for DenseNet121
        self.head     = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)   # (B, 1024)
        return self.head(feats)    # (B, num_classes)

model = DenseNet121Classifier(model, N_CLASSES).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params= sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ DenseNet121 model ready")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")
print(f"  Feature dim      : {model.backbone.num_features}")
print(f"  Output classes   : {N_CLASSES}")

# Quick forward pass test
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
    out   = model(dummy)
print(f"  Output shape     : {out.shape}  ← (batch=2, classes={N_CLASSES}) ✓")


## Cell 8 — Focal Loss + Optimizer + Scheduler

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss: down-weights easy examples, focuses on hard ones.
       Especially useful for imbalanced medical datasets."""
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight  = weight
        self.gamma   = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets,
                                  weight=self.weight, reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = ((1 - pt) ** self.gamma) * ce_loss
        return focal.mean()

# ── Loss: Focal (gamma=2) + class weights ─────────────────
criterion = FocalLoss(weight=class_weights, gamma=2.0)

# ── Optimizer: AdamW ──────────────────────────────────────
# DenseNet benefits from slightly lower LR than ResNet
optimizer = AdamW(
    model.parameters(),
    lr           = LR,          # 1e-4
    weight_decay = WEIGHT_DECAY, # 1e-2
)

# ── Scheduler: Cosine Annealing ───────────────────────────
scheduler = CosineAnnealingLR(
    optimizer,
    T_max  = N_EPOCHS,
    eta_min= 1e-6
)

# ── Mixed precision scaler ────────────────────────────────
scaler = GradScaler()

print("✓ Loss / Optimizer / Scheduler ready")
print(f"  Loss      : FocalLoss (gamma=2.0) + class weights")
print(f"  Optimizer : AdamW (lr={LR}, wd={WEIGHT_DECAY})")
print(f"  Scheduler : CosineAnnealingLR (T_max={N_EPOCHS}, eta_min=1e-6)")
print(f"  AMP       : GradScaler (FP16 mixed precision)")


## Cell 9 — Training & Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, grad_accum):
    model.train()
    total_loss = 0.0
    all_preds  = []
    all_labels = []
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(device)
        labels = labels.to(device)

        with autocast():
            outputs = model(imgs)
            loss    = criterion(outputs, labels) / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        preds       = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    return total_loss / len(loader), acc


@torch.no_grad()
def evaluate(model, loader, criterion, device, n_classes):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []
    all_probs  = []

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        with autocast():
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        probs  = torch.softmax(outputs, dim=1).cpu().numpy()
        preds  = outputs.argmax(dim=1).cpu().numpy()

        total_loss  += loss.item()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc = (all_preds == all_labels).mean()
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    try:
        auc = roc_auc_score(
            all_labels, all_probs,
            multi_class='ovr', average='macro'
        )
    except ValueError:
        auc = 0.0

    return total_loss / len(loader), acc, f1, auc, all_preds, all_labels, all_probs


print("✓ train_one_epoch() and evaluate() functions ready")


## Cell 10 — TRAIN DenseNet121 (50 Epochs)
> Expected: ~2 min/epoch on T4 → ~100 min total

In [ ]:
CKPT_PATH = f"{OUTPUT}{MODEL_NAME}_best.pth"

best_val_acc = 0.0
best_val_auc = 0.0
no_improve   = 0
history      = []

print("=" * 62)
print(f"  DENSENET121 TRAINING  —  {N_EPOCHS} epochs")
print(f"  Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print("=" * 62)
print(f"{'Ep':>3} {'TrLoss':>8} {'TrAcc':>7} {'VaLoss':>8} "
      f"{'VaAcc':>7} {'VaF1':>7} {'VaAUC':>7}  Status")
print("-" * 62)

train_start = time.time()

for epoch in range(N_EPOCHS):
    ep_start = time.time()

    # ── Train ──────────────────────────────────────────────
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion,
        scaler, DEVICE, GRAD_ACCUM
    )

    # ── Validate ───────────────────────────────────────────
    va_loss, va_acc, va_f1, va_auc, _, _, _ = evaluate(
        model, val_loader, criterion, DEVICE, N_CLASSES
    )

    scheduler.step()
    ep_time = time.time() - ep_start

    # ── Checkpoint ─────────────────────────────────────────
    improved = False
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_val_auc = va_auc
        torch.save(model.state_dict(), CKPT_PATH)
        no_improve = 0
        improved   = True
    else:
        no_improve += 1

    status = f"⭐ BEST acc={best_val_acc:.4f}" if improved else f"no imp {no_improve}/{PATIENCE}"

    print(f"{epoch+1:>3} {tr_loss:>8.4f} {tr_acc:>7.4f} {va_loss:>8.4f} "
          f"{va_acc:>7.4f} {va_f1:>7.4f} {va_auc:>7.4f}  {status}  "
          f"[{ep_time:.0f}s]")

    history.append({
        'epoch'   : epoch + 1,
        'tr_loss' : tr_loss,
        'tr_acc'  : tr_acc,
        'va_loss' : va_loss,
        'va_acc'  : va_acc,
        'va_f1'   : va_f1,
        'va_auc'  : va_auc,
    })

    # ── Early stopping ─────────────────────────────────────
    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

total_time = time.time() - train_start
print("=" * 62)
print(f"Training done in {total_time/60:.1f} min")
print(f"Best Val Acc : {best_val_acc:.4f}")
print(f"Best Val AUC : {best_val_auc:.4f}")
print(f"Checkpoint   : {CKPT_PATH}")
print("=" * 62)

# Save history JSON
with open(f"{OUTPUT}{MODEL_NAME}_history.json", 'w') as f:
    json.dump(history, f, indent=2)
print("✓ History saved")


## Cell 11 — Training Curves Plot

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('DenseNet121 — Training Curves', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(hist_df['epoch'], hist_df['tr_loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(hist_df['epoch'], hist_df['va_loss'], label='Val Loss',   color='#2ecc71')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(hist_df['epoch'], hist_df['tr_acc'], label='Train Acc', color='#e74c3c')
axes[1].plot(hist_df['epoch'], hist_df['va_acc'], label='Val Acc',   color='#2ecc71')
axes[1].axhline(y=best_val_acc, color='gold', linestyle='--', label=f'Best {best_val_acc:.4f}')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

# AUC
axes[2].plot(hist_df['epoch'], hist_df['va_auc'], label='Val AUC', color='#3498db')
axes[2].axhline(y=best_val_auc, color='gold', linestyle='--', label=f'Best {best_val_auc:.4f}')
axes[2].set_title('AUC-ROC'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {MODEL_NAME}_curves.png")


## Cell 12 — Final Test Evaluation

In [ ]:
# Load best checkpoint
print("Loading best checkpoint...")
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

# Evaluate on test set
ts_loss, ts_acc, ts_f1, ts_auc, ts_preds, ts_labels, ts_probs = evaluate(
    model, test_loader, criterion, DEVICE, N_CLASSES
)

print("\n" + "=" * 58)
print(f"  DENSENET121 — FINAL TEST RESULTS")
print("=" * 58)
print(f"  Test Accuracy : {ts_acc*100:.2f}%")
print(f"  Test F1 Macro : {ts_f1:.4f}")
print(f"  Test AUC-ROC  : {ts_auc:.4f}")
print(f"  Best Val Acc  : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC  : {best_val_auc:.4f}")
print("=" * 58)

print("\nPer-class Report:")
print(classification_report(
    ts_labels, ts_preds,
    target_names=CLASS_NAMES,
    digits=2
))

# Compare with previous models
print("\n" + "=" * 58)
print("  BENCHMARK COMPARISON")
print("=" * 58)
print(f"  {'Model':<20} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'VGG16 (baseline)':<20} {'80.48%':>8} {'0.7102':>8} {'0.9601':>8}")
print(f"  {'ResNet50':<20} {'87.40%':>8} {'0.7261':>8} {'0.9823':>8}")
print(f"  {'DenseNet121 ★':<20} {ts_acc*100:>7.2f}% {ts_f1:>8.4f} {ts_auc:>8.4f}")
print("=" * 58)


## Cell 13 — Confusion Matrix

In [ ]:
cm = confusion_matrix(ts_labels, ts_preds)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('DenseNet121 — Confusion Matrices', fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Raw Counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Normalized (Recall per Class)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_confusion.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {MODEL_NAME}_confusion.png")


## Cell 14 — Per-Class Bar Chart

In [ ]:
from sklearn.metrics import precision_score, recall_score

per_precision = precision_score(ts_labels, ts_preds, average=None, zero_division=0)
per_recall    = recall_score(ts_labels, ts_preds, average=None, zero_division=0)
per_f1        = f1_score(ts_labels, ts_preds, average=None, zero_division=0)

x    = np.arange(N_CLASSES)
w    = 0.25
fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(x - w, per_precision, w, label='Precision', color='#3498db', alpha=0.85)
ax.bar(x,     per_recall,    w, label='Recall',    color='#2ecc71', alpha=0.85)
ax.bar(x + w, per_f1,        w, label='F1',        color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('DenseNet121 — Per-Class Precision / Recall / F1')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for i, (p, r, f) in enumerate(zip(per_precision, per_recall, per_f1)):
    ax.text(i - w, p + 0.02, f'{p:.2f}', ha='center', va='bottom', fontsize=8)
    ax.text(i,     r + 0.02, f'{r:.2f}', ha='center', va='bottom', fontsize=8)
    ax.text(i + w, f + 0.02, f'{f:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_perclass.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {MODEL_NAME}_perclass.png")


## Cell 15 — Save Results JSON & Final Summary

In [ ]:
results = {
    'model'         : MODEL_NAME,
    'timm_str'      : TIMM_STR,
    'total_params'  : sum(p.numel() for p in model.parameters()),
    'test_accuracy' : float(ts_acc),
    'test_f1_macro' : float(ts_f1),
    'test_auc_roc'  : float(ts_auc),
    'best_val_acc'  : float(best_val_acc),
    'best_val_auc'  : float(best_val_auc),
    'epochs_run'    : len(history),
    'n_classes'     : N_CLASSES,
    'class_names'   : CLASS_NAMES,
    'per_class' : {
        name: {
            'precision': float(per_precision[i]),
            'recall'   : float(per_recall[i]),
            'f1'       : float(per_f1[i]),
        }
        for i, name in enumerate(CLASS_NAMES)
    },
    'training_history': history,
}

result_path = f"{OUTPUT}{MODEL_NAME}_results.json"
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)

print("=" * 58)
print(f"  ✓ Results saved to: {MODEL_NAME}_results.json")
print(f"  ✓ Model saved to  : {MODEL_NAME}_best.pth")
print(f"  ✓ Curves saved    : {MODEL_NAME}_curves.png")
print(f"  ✓ Confusion saved : {MODEL_NAME}_confusion.png")
print("=" * 58)
print("\n>>> DOWNLOAD THESE FROM OUTPUT TAB BEFORE CLOSING <<<")
print(f"  ! {MODEL_NAME}_best.pth")
print(f"  ! {MODEL_NAME}_results.json")
print(f"  ! {MODEL_NAME}_curves.png")
print(f"  ! {MODEL_NAME}_confusion.png")
print("=" * 58)
print("\nBENCHMARK UPDATE:")
print(f"  VGG16      →  80.48%  F1=0.7102  AUC=0.9601")
print(f"  ResNet50   →  87.40%  F1=0.7261  AUC=0.9823")
print(f"  DenseNet121→  {ts_acc*100:.2f}%  F1={ts_f1:.4f}  AUC={ts_auc:.4f}")

!ls -lh /kaggle/working/*.pth /kaggle/working/*.json /kaggle/working/*.png 2>/dev/null
